# 04 Staging — DLT Pipeline (Expectations + graph view)

**Doel:** Lees `ORDER_HEADER` en `ORDER_DETAIL` parquet-bestanden via Auto Loader en
schrijf ze als Delta Live Tables naar `STAGING_AZURESTORAGE` met:

- **Expectations** op drie ernstniveaus om datakwaliteit zichtbaar te maken
- **Declaratieve graph view** in de DLT-UI (twee nodes: `order_header_dlt` en `order_detail_dlt`)
- Dezelfde **vijf audit-kolommen** als de basis-pipeline

| Expectation | Actie bij schending |
|---|---|
| `expect` | Rij wordt opgeslagen, teller loopt op |
| `expect_or_drop` | Rij wordt weggegooid, teller loopt op |
| `expect_or_fail` | Pipeline stopt bij schending |

**Parameterisatie:** Alleen `catalog` — de DLT-pipeline leest **niet** uit de control table.
Het bronpad wordt afgeleid van `catalog`: `/Volumes/{catalog}/staging_azurestorage/source`.

| Audit-kolom | Bron |
|---|---|
| `_ingestion_timestamp` | `current_timestamp()` |
| `_source_system` | literal `azurestorage` |
| `_source_file` | `_metadata.file_path` |
| `_last_modified` | `_metadata.file_modification_time` |
| `_pipeline_run_id` | `spark.conf.get("pipelines.id")` |

In [ ]:
# Parameters & widgets
# In een DLT-pipeline worden widgets gelezen via spark.conf.get.
# De DAB resource injecteert 'catalog' als pipeline-configuratieparameter.
import dlt
from pyspark.sql import functions as F

catalog = spark.conf.get("catalog", "DEMO_DEV")
source_path = f"/Volumes/{catalog.lower()}/staging_azurestorage/source"
pipeline_run_id = spark.conf.get("pipelines.id", "")

print(f"Catalog      : {catalog}")
print(f"Source path  : {source_path}")
print(f"Pipeline id  : {pipeline_run_id}")

## Tabel 1 — `order_header_dlt`

Expectations:
1. `expect` — `order_id IS NOT NULL` (logt schendingen, behoudt rij)
2. `expect_or_drop` — `order_date IS NOT NULL` (verwijdert rij bij schending)
3. `expect_or_fail` — `customer_id IS NOT NULL` (stopt pipeline bij schending)

In [ ]:
@dlt.expect("order_id_not_null", "order_id IS NOT NULL")
@dlt.expect_or_drop("order_date_not_null", "order_date IS NOT NULL")
@dlt.expect_or_fail("customer_id_not_null", "customer_id IS NOT NULL")
@dlt.table(
    name="order_header_dlt",
    comment="Staging tabel voor ORDER_HEADER parquet-bestanden (DLT-versie met Expectations)",
)
def order_header_dlt():
    raw_df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("pathGlobFilter", "ORDER_HEADER_*.parquet")
        .option("cloudFiles.inferColumnTypes", "true")
        .load(source_path)
    )

    return (
        raw_df
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_source_system",       F.lit("azurestorage"))
        .withColumn("_source_file",         F.col("_metadata.file_path"))
        .withColumn("_last_modified",       F.col("_metadata.file_modification_time"))
        .withColumn("_pipeline_run_id",     F.lit(pipeline_run_id))
        .drop("_metadata")
    )

## Tabel 2 — `order_detail_dlt`

Expectations:
1. `expect` — `order_id IS NOT NULL` (logt schendingen, behoudt rij)
2. `expect_or_drop` — `product_id IS NOT NULL` (verwijdert rij bij schending)
3. `expect_or_fail` — `quantity IS NOT NULL` (stopt pipeline bij schending)

In [ ]:
@dlt.expect("order_id_not_null", "order_id IS NOT NULL")
@dlt.expect_or_drop("product_id_not_null", "product_id IS NOT NULL")
@dlt.expect_or_fail("quantity_not_null", "quantity IS NOT NULL")
@dlt.table(
    name="order_detail_dlt",
    comment="Staging tabel voor ORDER_DETAIL parquet-bestanden (DLT-versie met Expectations)",
)
def order_detail_dlt():
    raw_df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("pathGlobFilter", "ORDER_DETAIL_*.parquet")
        .option("cloudFiles.inferColumnTypes", "true")
        .load(source_path)
    )

    return (
        raw_df
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_source_system",       F.lit("azurestorage"))
        .withColumn("_source_file",         F.col("_metadata.file_path"))
        .withColumn("_last_modified",       F.col("_metadata.file_modification_time"))
        .withColumn("_pipeline_run_id",     F.lit(pipeline_run_id))
        .drop("_metadata")
    )